In [ ]:
import os
import random
import math
import numpy as np
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True 
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
import json
import gc

2.10.0+cu128
12.8
Tesla T4
(7, 5)


In [ ]:
EXPERIMENT_ID = "baseline"
 
TIMESTEPS = 200          # {100, 200, 400}
BASE_CHANNELS = 32       # {16, 32, 64}
LEARNING_RATE = 1e-4     # {1e-5, 1e-4, 1e-3}

NUM_EPOCHS = 275
BATCH_SIZE = 64
LATENT_DIM = None  
IMAGE_SIZE = 64
NUM_SEEDS = 3
 
FID_NUM_SAMPLES = 2048  # how many images to generate for FID
SAVE_SAMPLES_EVERY = 25  # save sample grids every N epochs
CHECKPOINT_EVERY = 50
 
_possible_paths = [
    "/kaggle/input/datasets/crawford/cat-dataset/cats"
]
DATASET_PATH = None
for p in _possible_paths:
    if os.path.exists(p):
        DATASET_PATH = p
        break
if DATASET_PATH is None:
    print("Available inputs:", os.listdir("/kaggle/input/"))
    raise FileNotFoundError("Cat dataset not found - adjust DATASET_PATH")
 
OUTPUT_DIR = f"/kaggle/working/ddpm_{EXPERIMENT_ID}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output: {OUTPUT_DIR}")

Output: /kaggle/working/ddpm_baseline


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cuda


In [ ]:
class CatDataset(Dataset):
    """Loads cat images from the dataset directory."""
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.bad_indices = set()
 
        # Collect all image files — skip annotation files (.cat)
        self.image_paths = []
        for ext in ["*.jpg", "*.jpeg", "*.png"]:
            self.image_paths.extend(list(self.root_dir.rglob(ext)))
        self.image_paths = sorted(self.image_paths)
        print(f"Found {len(self.image_paths)} images in {self.root_dir}")
 
        # Quick sanity: verify a few images load
        n_check = min(20, len(self.image_paths))
        n_ok = 0
        for i in range(n_check):
            try:
                Image.open(self.image_paths[i]).convert("RGB")
                n_ok += 1
            except Exception:
                pass
        print(f"Sanity check: {n_ok}/{n_check} images loaded OK")
        if n_ok == 0:
            raise RuntimeError("No images could be loaded — check dataset path!")
 
    def __len__(self):
        return len(self.image_paths)
 
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            image.load()  # force full decode to catch truncation
        except Exception:
            self.bad_indices.add(idx)
            # Return a random valid image
            fallback = random.randint(0, len(self) - 1)
            while fallback in self.bad_indices:
                fallback = random.randint(0, len(self) - 1)
            return self.__getitem__(fallback)
        if self.transform:
            image = self.transform(image)
        return image
 
 
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # -> [-1, 1]
])
 
print(f"Dataset path: {DATASET_PATH}")
print(f"Contents: {os.listdir(DATASET_PATH)[:15]}...")  
 
dataset = CatDataset(DATASET_PATH, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=4, pin_memory=True, drop_last=True)
 
# Quick sanity check
sample_batch = next(iter(dataloader))
print(f"Batch shape: {sample_batch.shape}")  # [B, 3, 64, 64]

# Show a few real images
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    img = sample_batch[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[i].imshow(img.clip(0, 1))
    axes[i].axis("off")
plt.suptitle("Real cat images (64×64)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/real_samples.png", dpi=100)
plt.show()


Dataset path: /kaggle/input/datasets/crawford/cat-dataset/cats
Contents: ['CAT_03', 'CAT_02', 'CAT_00', 'CAT_01', 'CAT_06', 'CAT_04', 'CAT_05']...


In [ ]:
def cosine_beta_schedule(timesteps, s=0.008):
    """Cosine schedule as proposed in Improved DDPM (Nichol & Dhariwal, 2021)."""
    steps = timesteps + 1
    t = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((t / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clamp(betas, 0.0001, 0.9999)

def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    scale = 1000 / timesteps
    beta_start = beta_start * scale
    beta_end = beta_end * scale
    return torch.linspace(beta_start, beta_end, timesteps)
 
 
class DiffusionSchedule:
    """Precompute all diffusion quantities."""
    def __init__(self, timesteps, device):
        self.timesteps = timesteps
        betas = linear_beta_schedule(timesteps).to(device) 
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
 
        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.alphas_cumprod_prev = alphas_cumprod_prev
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
 
        self.posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
 
    def q_sample(self, x_start, t, noise=None):
        """Forward process: add noise to x_start at timestep t."""
        if noise is None:
            noise = torch.randn_like(x_start)
        sqrt_alpha = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_alpha * x_start + sqrt_one_minus * noise, noise
 
 
schedule = DiffusionSchedule(TIMESTEPS, device)


In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        # Pre-norm
        h = self.norm(x)
        
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)
        
        q = q.view(B, C, -1)
        k = k.view(B, C, -1)
        v = v.view(B, C, -1)

        attn = torch.einsum("bci,bcj->bij", q, k) * (C ** -0.5)
        attn = F.softmax(attn, dim=-1)

        out = torch.einsum("bij,bcj->bci", attn, v)
        out = out.view(B, C, H, W)
        
        return x + self.proj(out)

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
 
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb
 
 
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.bn1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch),
        )
        
        self.bn2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.residual_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.dropout = nn.Dropout(p=0.1)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.bn1(x)))
        
        t = self.time_mlp(t_emb)[:, :, None, None]
        h = h + t
        
        h = self.conv2(F.silu(self.bn2(h)))
        h = self.dropout(h)
        return h + self.residual_conv(x)
 
class UNet(nn.Module):
    def __init__(self, base_channels=32, time_emb_dim=256):
        super().__init__()
        ch = base_channels
        
        self.time_emb = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim * 2),
        )
        t_dim = time_emb_dim * 2

        self.conv_in = nn.Conv2d(3, ch, 3, padding=1)

        # --- ENCODER ---
        # Level 1: 64x64
        self.down1a = ResidualBlock(ch, ch, t_dim)
        self.down1b = ResidualBlock(ch, ch, t_dim)
        self.pool1 = nn.Conv2d(ch, ch, 4, stride=2, padding=1)

        # Level 2: 32x32
        self.down2a = ResidualBlock(ch, 2*ch, t_dim)
        self.down2b = ResidualBlock(2*ch, 2*ch, t_dim)
        self.pool2 = nn.Conv2d(2*ch, 2*ch, 4, stride=2, padding=1)

        # Level 3: 16x16 + ATTENTION
        self.down3a = ResidualBlock(2*ch, 4*ch, t_dim)
        self.attn3a = AttentionBlock(4*ch)
        self.down3b = ResidualBlock(4*ch, 4*ch, t_dim)
        self.attn3b = AttentionBlock(4*ch)
        self.pool3 = nn.Conv2d(4*ch, 4*ch, 4, stride=2, padding=1)

        # Level 4: 8x8
        self.down4a = ResidualBlock(4*ch, 4*ch, t_dim)
        self.down4b = ResidualBlock(4*ch, 4*ch, t_dim)

        # --- BOTTLENECK (8x8) ---
        self.mid1 = ResidualBlock(4*ch, 4*ch, t_dim)
        self.mid_attn = AttentionBlock(4*ch)
        self.mid2 = ResidualBlock(4*ch, 4*ch, t_dim)

        # --- DECODER ---
        # Level 4: 8x8 -> 16x16
        self.up4 = nn.ConvTranspose2d(4*ch, 4*ch, 4, stride=2, padding=1)
        self.up_res4a = ResidualBlock(8*ch, 4*ch, t_dim) 
        self.up_res4b = ResidualBlock(4*ch, 4*ch, t_dim)

        # Level 3: 16x16 -> 32x32 + ATTENTION
        self.up3 = nn.ConvTranspose2d(4*ch, 2*ch, 4, stride=2, padding=1)
        self.up_res3a = ResidualBlock(4*ch, 2*ch, t_dim) 
        self.up_attn3 = AttentionBlock(2*ch)
        self.up_res3b = ResidualBlock(2*ch, 2*ch, t_dim)

        # Level 2: 32x32 -> 64x64
        self.up2 = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1)
        self.up_res2a = ResidualBlock(2*ch, ch, t_dim)
        self.up_res2b = ResidualBlock(ch, ch, t_dim)

        # Output (64x64)
        self.conv_out = nn.Sequential(
            nn.GroupNorm(8, ch),
            nn.SiLU(),
            nn.Conv2d(ch, 3, 3, padding=1),
        )
        
        nn.init.zeros_(self.conv_out[-1].weight)
        nn.init.zeros_(self.conv_out[-1].bias)

    def forward(self, x, t):
        t_emb = self.time_emb(t)
        
        x0 = self.conv_in(x)

        # Encoder
        d1 = self.down1b(self.down1a(x0, t_emb), t_emb)
        d1_p = self.pool1(d1)
        
        d2 = self.down2b(self.down2a(d1_p, t_emb), t_emb)
        d2_p = self.pool2(d2)

        d3 = self.attn3b(self.down3b(self.attn3a(self.down3a(d2_p, t_emb)), t_emb))
        d3_p = self.pool3(d3)

        d4 = self.down4b(self.down4a(d3_p, t_emb), t_emb)

        # Bottleneck
        m = self.mid1(d4, t_emb)
        m = self.mid_attn(m)
        m = self.mid2(m, t_emb)

        # Decoder
        u4 = self.up4(m)
        u4 = self.up_res4b(self.up_res4a(torch.cat([u4, d3], dim=1), t_emb), t_emb)

        u3 = self.up3(u4)
        u3 = self.up_res3b(self.up_attn3(self.up_res3a(torch.cat([u3, d2], dim=1), t_emb)), t_emb)

        u2 = self.up2(u3)
        u2 = self.up_res2b(self.up_res2a(torch.cat([u2, d1], dim=1), t_emb), t_emb)

        return self.conv_out(u2)

 
model = UNet(base_channels=64).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")


Model parameters: 4,996,227


In [ ]:
import copy

class EMA:
    def __init__(self, beta=0.9999):
        self.beta = beta

    def update_model_average(self, ema_model, current_model):
        for current_params, ema_params in zip(current_model.parameters(), ema_model.parameters()):
            old_weight, up_weight = ema_params.data, current_params.data
            ema_params.data = self.update_average(old_weight, up_weight)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1 - self.beta) * new

In [9]:
def train_one_epoch(model, ema_model, ema_helper, dataloader, optimizer, schedule, device, scaler):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        batch = batch.to(device)
        B = batch.shape[0]

        t = torch.randint(0, schedule.timesteps, (B,), device=device)
        noise = torch.randn_like(batch)
        x_noisy, _ = schedule.q_sample(batch, t, noise=noise)

        with torch.cuda.amp.autocast():
            noise_pred = model(x_noisy, t)
            loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        ema_helper.update_model_average(ema_model, model)

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches

In [ ]:
@torch.no_grad()
def sample_ddpm(model, schedule, n_samples, device, image_size=64):
    model.eval()
    x = torch.randn(n_samples, 3, image_size, image_size, device=device)
 
    for t_idx in tqdm(reversed(range(schedule.timesteps)), total=schedule.timesteps,
                      desc="Sampling", leave=False):
        t = torch.full((n_samples,), t_idx, device=device, dtype=torch.long)
        noise_pred = model(x, t)
 
        alpha = schedule.alphas[t_idx]
        alpha_cumprod = schedule.alphas_cumprod[t_idx]
        beta = schedule.betas[t_idx]
 
        mean = (1.0 / alpha.sqrt()) * (
            x - (beta / (1.0 - alpha_cumprod).sqrt()) * noise_pred
        )
 
        if t_idx > 0:
            noise = torch.randn_like(x)
            sigma = schedule.posterior_variance[t_idx].sqrt()
            x = mean + sigma * noise
        else:
            x = mean
 
    x = x.clamp(-1, 1) * 0.5 + 0.5
    return x
 
 
@torch.no_grad()
def sample_ddpm_with_latents(model, schedule, n_samples, device, image_size=64):
    """Sample and return both images and initial noise (for interpolation)."""
    model.eval()
    z_init = torch.randn(n_samples, 3, image_size, image_size, device=device)
    x = z_init.clone()
 
    for t_idx in reversed(range(schedule.timesteps)):
        t = torch.full((n_samples,), t_idx, device=device, dtype=torch.long)
        noise_pred = model(x, t)
 
        alpha = schedule.alphas[t_idx]
        alpha_cumprod = schedule.alphas_cumprod[t_idx]
        beta = schedule.betas[t_idx]
 
        mean = (1.0 / alpha.sqrt()) * (
            x - (beta / (1.0 - alpha_cumprod).sqrt()) * noise_pred
        )
 
        if t_idx > 0:
            noise = torch.randn_like(x)
            sigma = schedule.posterior_variance[t_idx].sqrt()
            x = mean + sigma * noise
        else:
            x = mean
 
    x = x.clamp(-1, 1) * 0.5 + 0.5
    return x, z_init
 
 
def save_sample_grid(model, schedule, device, epoch, output_dir, n=64):
    """Generate and save a grid of samples."""
    samples = sample_ddpm(model, schedule, n, device)
    grid = make_grid(samples, nrow=8, padding=2)
    save_image(grid, f"{output_dir}/samples_epoch_{epoch:04d}.png")
    return samples


In [ ]:
import subprocess
try:
    import pytorch_fid
except ImportError:
    subprocess.check_call(["pip", "install", "pytorch-fid", "-q"])
 
from pytorch_fid import fid_score
 
def compute_fid(model, schedule, device, real_image_dir, output_dir,
                n_samples=2048, batch_size=64):
    """Generate images and compute FID against real images."""
    gen_dir = os.path.join(output_dir, "fid_generated")
    os.makedirs(gen_dir, exist_ok=True)
 
    model.eval()
    generated = 0
    while generated < n_samples:
        n = min(batch_size, n_samples - generated)
        samples = sample_ddpm(model, schedule, n, device)
        for i in range(n):
            save_image(samples[i], f"{gen_dir}/{generated + i:05d}.png")
        generated += n
 
    real_dir = os.path.join(output_dir, "fid_real")
    if not os.path.exists(real_dir) or len(os.listdir(real_dir)) == 0:
        os.makedirs(real_dir, exist_ok=True)
        ds_no_aug = CatDataset(DATASET_PATH, transform=transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
        ]))
        for i in range(min(len(ds_no_aug), n_samples)):
            img = ds_no_aug[i]
            save_image(img, f"{real_dir}/{i:05d}.png")
 
    fid_value = fid_score.calculate_fid_given_paths(
        [real_dir, gen_dir],
        batch_size=50,
        device=device,
        dims=2048,
    )
 
    import shutil
    shutil.rmtree(gen_dir)
 
    return fid_value


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 115.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [ ]:
def run_experiment(seed, experiment_id, timesteps, base_channels, lr, output_dir):
    """Run one full training run with given seed and hyperparameters."""
    set_seed(seed)
    
    exp_dir = f"{output_dir}/seed_{seed}"
    os.makedirs(exp_dir, exist_ok=True)

    # Model & optimizer
    sched = DiffusionSchedule(timesteps, device)
    mdl = UNet(base_channels=base_channels).to(device)
    optimizer = torch.optim.Adam(mdl.parameters(), lr=lr)

    ema = EMA(beta=0.9999)
    ema_model = copy.deepcopy(mdl).eval() 
    ema_model.requires_grad_(False)       

    n_params = sum(p.numel() for p in mdl.parameters())
    print(f"\n{'='*60}")
    print(f"Seed={seed} | T={timesteps} | ch={base_channels} | lr={lr}")
    print(f"Parameters: {n_params:,}")
    print(f"{'='*60}")

    history = {"epoch": [], "loss": [], "fid": []}

    best_fid = float("inf")
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(1, NUM_EPOCHS + 1):
        avg_loss = train_one_epoch(mdl, ema_model, ema, dataloader, optimizer, sched, device, scaler)
        history["epoch"].append(epoch)
        history["loss"].append(avg_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.5f}")

        if epoch % SAVE_SAMPLES_EVERY == 0 or epoch == NUM_EPOCHS:
            save_sample_grid(ema_model, sched, device, epoch, exp_dir)
            print(f"  -> Saved sample grid at epoch {epoch}")

        if epoch == NUM_EPOCHS:
            fid = compute_fid(ema_model, sched, device, DATASET_PATH, exp_dir,
                              n_samples=FID_NUM_SAMPLES)
            history["fid"].append({"epoch": epoch, "fid": fid})
            print(f"  -> FID: {fid:.2f}")

            if fid < best_fid:
                best_fid = fid
                torch.save(ema_model.state_dict(), f"{exp_dir}/best_model.pt")
                print(f"  -> New best model saved (FID={fid:.2f})")

        if epoch % CHECKPOINT_EVERY == 0:
            torch.save({
                "epoch": epoch,
                "model_state_dict": mdl.state_dict(),
                "ema_model_state_dict": ema_model.state_dict(), 
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": avg_loss,
            }, f"{exp_dir}/checkpoint_epoch_{epoch}.pt")

    torch.save(ema_model.state_dict(), f"{exp_dir}/final_model.pt")

    with open(f"{exp_dir}/history.json", "w") as f:
        json.dump(history, f, indent=2)

    plt.figure(figsize=(10, 4))
    plt.plot(history["epoch"], history["loss"])
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title(f"Training Loss (seed={seed}, T={timesteps}, ch={base_channels}, lr={lr})")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{exp_dir}/loss_curve.png", dpi=100)
    plt.show()

    return history, ema_model, sched


# Experiments

## Timesteps

In [ ]:
SEEDS = [42, 142, 242]

phase1_configs = [
    {"timesteps": 100,  "base_channels": 32, "lr": 1e-4},
    {"timesteps": 200,  "base_channels": 32, "lr": 1e-4},  # baseline
    {"timesteps": 400,  "base_channels": 32, "lr": 1e-4},
]

PHASE1_IDX = 0

cfg = phase1_configs[PHASE1_IDX]
cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"

print(f"\n{'='*60}")
print(f"CONFIG: {cfg_name}")
print(f"{'='*60}")

fid_scores = []
for seed in SEEDS:
    output_dir = f"/kaggle/working/ddpm_{cfg_name}"
    history, _, _ = run_experiment(
        seed=seed,
        experiment_id=cfg_name,
        timesteps=cfg["timesteps"],
        base_channels=cfg["base_channels"],
        lr=cfg["lr"],
        output_dir=output_dir,
    )
    final_fid = history["fid"][-1]["fid"]
    fid_scores.append(final_fid)
    print(f"  Seed {seed}: FID = {final_fid:.2f}")

mean_fid = np.mean(fid_scores)
std_fid = np.std(fid_scores)
print(f"\n=> {cfg_name}: FID = {mean_fid:.2f} ± {std_fid:.2f}")

In [ ]:
SEEDS = [42, 142, 242]

phase1_configs = [
    {"timesteps": 100,  "base_channels": 32, "lr": 1e-4},
    {"timesteps": 200,  "base_channels": 32, "lr": 1e-4},  # baseline
    {"timesteps": 400,  "base_channels": 32, "lr": 1e-4},
]

PHASE1_IDX = 1

cfg = phase1_configs[PHASE1_IDX]
cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"

print(f"\n{'='*60}")
print(f"CONFIG: {cfg_name}")
print(f"{'='*60}")

fid_scores = []
for seed in SEEDS:
    output_dir = f"/kaggle/working/ddpm_{cfg_name}"
    history, _, _ = run_experiment(
        seed=seed,
        experiment_id=cfg_name,
        timesteps=cfg["timesteps"],
        base_channels=cfg["base_channels"],
        lr=cfg["lr"],
        output_dir=output_dir,
    )
    final_fid = history["fid"][-1]["fid"]
    fid_scores.append(final_fid)
    print(f"  Seed {seed}: FID = {final_fid:.2f}")

mean_fid = np.mean(fid_scores)
std_fid = np.std(fid_scores)
print(f"\n=> {cfg_name}: FID = {mean_fid:.2f} ± {std_fid:.2f}")

In [ ]:
SEEDS = [42, 142, 242]

phase1_configs = [
    {"timesteps": 100,  "base_channels": 32, "lr": 1e-4},
    {"timesteps": 200,  "base_channels": 32, "lr": 1e-4}, 
    {"timesteps": 400,  "base_channels": 32, "lr": 1e-4},
]

PHASE1_IDX = 2

cfg = phase1_configs[PHASE1_IDX]
cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"

print(f"\n{'='*60}")
print(f"CONFIG: {cfg_name}")
print(f"{'='*60}")

fid_scores = []
for seed in SEEDS:
    output_dir = f"/kaggle/working/ddpm_{cfg_name}"
    history, _, _ = run_experiment(
        seed=seed,
        experiment_id=cfg_name,
        timesteps=cfg["timesteps"],
        base_channels=cfg["base_channels"],
        lr=cfg["lr"],
        output_dir=output_dir,
    )
    final_fid = history["fid"][-1]["fid"]
    fid_scores.append(final_fid)
    print(f"  Seed {seed}: FID = {final_fid:.2f}")

mean_fid = np.mean(fid_scores)
std_fid = np.std(fid_scores)
print(f"\n=> {cfg_name}: FID = {mean_fid:.2f} ± {std_fid:.2f}")

## Base channels

In [ ]:
SEEDS = [42, 142, 242]
phase2_configs = [
    {"timesteps": 200, "base_channels": 16, "lr": 1e-4},
    {"timesteps": 200, "base_channels": 32, "lr": 1e-4}, 
    {"timesteps": 200, "base_channels": 64, "lr": 1e-4},
]

PHASE2_IDX = 0

cfg = phase2_configs[PHASE2_IDX]
cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"
print(f"\n{'='*60}")
print(f"CONFIG: {cfg_name}")
print(f"{'='*60}")

fid_scores = []
for seed in SEEDS:
    output_dir = f"/kaggle/working/ddpm_{cfg_name}"
    history, _, _ = run_experiment(
        seed=seed,
        experiment_id=cfg_name,
        timesteps=cfg["timesteps"],
        base_channels=cfg["base_channels"],
        lr=cfg["lr"],
        output_dir=output_dir,
    )
    final_fid = history["fid"][-1]["fid"]
    fid_scores.append(final_fid)
    print(f"  Seed {seed}: FID = {final_fid:.2f}")

mean_fid = np.mean(fid_scores)
std_fid = np.std(fid_scores)
print(f"\n=> {cfg_name}: FID = {mean_fid:.2f} ± {std_fid:.2f}")

In [ ]:
SEEDS = [42, 142, 242]
phase2_configs = [
    {"timesteps": 200, "base_channels": 16, "lr": 1e-4},
    {"timesteps": 200, "base_channels": 32, "lr": 1e-4},  
    {"timesteps": 200, "base_channels": 64, "lr": 1e-4},
]

PHASE2_IDX = 2

cfg = phase2_configs[PHASE2_IDX]
cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"
print(f"\n{'='*60}")
print(f"CONFIG: {cfg_name}")
print(f"{'='*60}")

fid_scores = []
for seed in SEEDS:
    output_dir = f"/kaggle/working/ddpm_{cfg_name}"
    history, _, _ = run_experiment(
        seed=seed,
        experiment_id=cfg_name,
        timesteps=cfg["timesteps"],
        base_channels=cfg["base_channels"],
        lr=cfg["lr"],
        output_dir=output_dir,
    )
    final_fid = history["fid"][-1]["fid"]
    fid_scores.append(final_fid)
    print(f"  Seed {seed}: FID = {final_fid:.2f}")

mean_fid = np.mean(fid_scores)
std_fid = np.std(fid_scores)
print(f"\n=> {cfg_name}: FID = {mean_fid:.2f} ± {std_fid:.2f}")

## Learning rate

In [ ]:
for PHASE3_LR, PHASE3_SEEDS in [(1e-5, [242]), (1e-3, [42])]:
    cfg = {"timesteps": 200, "base_channels": 64, "lr": PHASE3_LR}
    cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"
    print(f"\n{'='*60}\nCONFIG: {cfg_name}\n{'='*60}")

    for seed in PHASE3_SEEDS:
        output_dir = f"/kaggle/working/ddpm_{cfg_name}"
        history, _, _ = run_experiment(seed=seed, experiment_id=cfg_name,
            timesteps=cfg["timesteps"], base_channels=cfg["base_channels"],
            lr=cfg["lr"], output_dir=output_dir)
        print(f"  Seed {seed}: FID = {history['fid'][-1]['fid']:.2f}")

In [ ]:
PHASE3_LR = 1e-5
PHASE3_SEEDS = [42, 142]

cfg = {"timesteps": 200, "base_channels": 64, "lr": PHASE3_LR}
cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"
print(f"\n{'='*60}\nCONFIG: {cfg_name}\n{'='*60}")

for seed in PHASE3_SEEDS:
    output_dir = f"/kaggle/working/ddpm_{cfg_name}"
    history, _, _ = run_experiment(seed=seed, experiment_id=cfg_name,
        timesteps=cfg["timesteps"], base_channels=cfg["base_channels"],
        lr=cfg["lr"], output_dir=output_dir)
    print(f"  Seed {seed}: FID = {history['fid'][-1]['fid']:.2f}")

In [ ]:
PHASE3_LR = 1e-3
PHASE3_SEEDS = [142, 242]

cfg = {"timesteps": 200, "base_channels": 64, "lr": PHASE3_LR}
cfg_name = f"T{cfg['timesteps']}_ch{cfg['base_channels']}_lr{cfg['lr']}"
print(f"\n{'='*60}\nCONFIG: {cfg_name}\n{'='*60}")

for seed in PHASE3_SEEDS:
    output_dir = f"/kaggle/working/ddpm_{cfg_name}"
    history, _, _ = run_experiment(seed=seed, experiment_id=cfg_name,
        timesteps=cfg["timesteps"], base_channels=cfg["base_channels"],
        lr=cfg["lr"], output_dir=output_dir)
    print(f"  Seed {seed}: FID = {history['fid'][-1]['fid']:.2f}")

# Cats and dogs experiment

In [ ]:
CATDOG_DIR = "/kaggle/input/datasets/bigdatamazur/cats-and-dogs/train/train"

catdog_dataset = CatDataset(CATDOG_DIR, transform=transform)
print(f"Dataset: {len(catdog_dataset.image_paths)} obrazów")
catdog_loader = DataLoader(catdog_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=4, pin_memory=True, drop_last=True)

transform_fid = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
catdog_fid = CatDataset(CATDOG_DIR, transform=transform_fid)
fid_real_dir = "/kaggle/working/ddpm_cats_dogs/seed_42/fid_real"
os.makedirs(fid_real_dir, exist_ok=True)
for i in range(min(len(catdog_fid), FID_NUM_SAMPLES)):
    save_image(catdog_fid[i], f"{fid_real_dir}/{i:05d}.png")
print(f"FID real: {min(len(catdog_fid), FID_NUM_SAMPLES)} obrazów gotowych")

dataloader = catdog_loader
history, _, _ = run_experiment(seed=42, experiment_id="cats_dogs",
    timesteps=200, base_channels=64, lr=1e-4,
    output_dir="/kaggle/working/ddpm_cats_dogs")
print(f"Seed 42 FID: {history['fid'][-1]['fid']:.2f}")

In [ ]:
CATDOG_DIR = "/kaggle/input/datasets/bolesawpaliwoda/cats-and-dogs/train"

catdog_dataset = CatDataset(CATDOG_DIR, transform=transform)
print(f"Dataset: {len(catdog_dataset.image_paths)} obrazów")
catdog_loader = DataLoader(catdog_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=4, pin_memory=True, drop_last=True)

transform_fid = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
catdog_fid = CatDataset(CATDOG_DIR, transform=transform_fid)
fid_real_dir = "/kaggle/working/ddpm_cats_dogs/seed_142/fid_real"
os.makedirs(fid_real_dir, exist_ok=True)
for i in range(min(len(catdog_fid), FID_NUM_SAMPLES)):
    save_image(catdog_fid[i], f"{fid_real_dir}/{i:05d}.png")
print(f"FID real: {min(len(catdog_fid), FID_NUM_SAMPLES)} of images are ready")

dataloader = catdog_loader
history, _, _ = run_experiment(seed=142, experiment_id="cats_dogs",
    timesteps=200, base_channels=64, lr=1e-4,
    output_dir="/kaggle/working/ddpm_cats_dogs")
print(f"Seed 142 FID: {history['fid'][-1]['fid']:.2f}")

In [ ]:
CATDOG_DIR = "/kaggle/input/datasets/mikoajrowicki/cats-and-dogs/train"

catdog_dataset = CatDataset(CATDOG_DIR, transform=transform)
print(f"Dataset: {len(catdog_dataset.image_paths)} obrazów")
catdog_loader = DataLoader(catdog_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=4, pin_memory=True, drop_last=True)

transform_fid = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
catdog_fid = CatDataset(CATDOG_DIR, transform=transform_fid)
fid_real_dir = "/kaggle/working/ddpm_cats_dogs/seed_242/fid_real"
os.makedirs(fid_real_dir, exist_ok=True)
for i in range(min(len(catdog_fid), FID_NUM_SAMPLES)):
    save_image(catdog_fid[i], f"{fid_real_dir}/{i:05d}.png")
print(f"FID real: {min(len(catdog_fid), FID_NUM_SAMPLES)} obrazów gotowych")

dataloader = catdog_loader
history, _, _ = run_experiment(seed=242, experiment_id="cats_dogs",
    timesteps=200, base_channels=64, lr=1e-4,
    output_dir="/kaggle/working/ddpm_cats_dogs")
print(f"Seed 242 FID: {history['fid'][-1]['fid']:.2f}")

In [ ]:
sched = DiffusionSchedule(200, device)
model = UNet(base_channels=64).to(device)
model.load_state_dict(torch.load("/kaggle/input/datasets/bolesawpaliwoda/best-model-cats/final_model.pt", map_location=device))
model.eval()

dataset = CatDataset(DATASET_PATH, transform=transform)

## Latent interpolation

In [ ]:
@torch.no_grad()
def latent_interpolation(model, schedule, device, n_pairs=50, n_interp=8, output_dir=OUTPUT_DIR):
    model.eval()
    save_dir = f"{output_dir}/interpolation"
    os.makedirs(save_dir, exist_ok=True)

    for pair_idx in range(n_pairs):
        _, z_init = sample_ddpm_with_latents(model, schedule, 2, device)
        z1, z2 = z_init[0:1], z_init[1:2]
        alphas = torch.linspace(0, 1, n_interp + 2, device=device)[1:-1]
        z_interp = torch.cat([z1] + [a * z2 + (1 - a) * z1 for a in alphas] + [z2], dim=0)

        x = z_interp.clone()
        for t_idx in reversed(range(schedule.timesteps)):
            t = torch.full((z_interp.shape[0],), t_idx, device=device, dtype=torch.long)
            noise_pred = model(x, t)
            mean = (1.0 / schedule.alphas[t_idx].sqrt()) * (
                x - (schedule.betas[t_idx] / (1.0 - schedule.alphas_cumprod[t_idx]).sqrt()) * noise_pred
            )
            if t_idx > 0:
                x = mean + schedule.posterior_variance[t_idx].sqrt() * torch.randn_like(x)
            else:
                x = mean
        x = x.clamp(-1, 1) * 0.5 + 0.5

        fig, axes = plt.subplots(1, n_interp + 2, figsize=(2 * (n_interp + 2), 2))
        labels = ["Start"] + [f"λ={a:.2f}" for a in alphas.cpu()] + ["End"]
        for col in range(n_interp + 2):
            axes[col].imshow(x[col].cpu().permute(1, 2, 0).numpy().clip(0, 1))
            axes[col].set_title(labels[col], fontsize=8)
            axes[col].axis("off")
        plt.suptitle(f"Interpolation - pair {pair_idx + 1}", fontsize=10)
        plt.tight_layout()
        plt.savefig(f"{save_dir}/pair_{pair_idx + 1:02d}.png", dpi=150)
        plt.close()
        print(f"Saved pair {pair_idx + 1}")

latent_interpolation(model, sched, device)

Saved pair 1
Saved pair 2
Saved pair 3
Saved pair 4
Saved pair 5
Saved pair 6
Saved pair 7
Saved pair 8
Saved pair 9
Saved pair 10
Saved pair 11
Saved pair 12
Saved pair 13
Saved pair 14
Saved pair 15
Saved pair 16
Saved pair 17
Saved pair 18
Saved pair 19
Saved pair 20
Saved pair 21
Saved pair 22
Saved pair 23
Saved pair 24
Saved pair 25
Saved pair 26
Saved pair 27
Saved pair 28
Saved pair 29
Saved pair 30
Saved pair 31
Saved pair 32
Saved pair 33
Saved pair 34
Saved pair 35
Saved pair 36
Saved pair 37
Saved pair 38
Saved pair 39
Saved pair 40
Saved pair 41
Saved pair 42
Saved pair 43
Saved pair 44
Saved pair 45
Saved pair 46
Saved pair 47
Saved pair 48
Saved pair 49
Saved pair 50


## Inpainting

In [ ]:
@torch.no_grad()
def repaint_inpainting(model, schedule, original_img, mask, device):
    model.eval()
    x = torch.randn_like(original_img)
    for t_idx in reversed(range(schedule.timesteps)):
        t = torch.full((1,), t_idx, device=device, dtype=torch.long)
        noise_pred = model(x, t)
        mean = (1.0 / schedule.alphas[t_idx].sqrt()) * (
            x - (schedule.betas[t_idx] / (1.0 - schedule.alphas_cumprod[t_idx]).sqrt()) * noise_pred
        )
        if t_idx > 0:
            x_denoised = mean + schedule.posterior_variance[t_idx].sqrt() * torch.randn_like(x)
            x_known, _ = schedule.q_sample(original_img, t, noise=torch.randn_like(original_img))
        else:
            x_denoised = mean
            x_known = original_img
        x = mask * x_known + (1 - mask) * x_denoised
    return x.clamp(-1, 1) * 0.5 + 0.5

def run_inpainting_demo(model, schedule, dataset, device, n_examples=50, mask_size=24, output_dir=OUTPUT_DIR):
    model.eval()
    save_dir = f"{output_dir}/inpainting"
    os.makedirs(save_dir, exist_ok=True)

    for i in tqdm(range(n_examples), desc="Inpainting"):
        img = dataset[i * (len(dataset) // n_examples)].unsqueeze(0).to(device)
        mask = torch.ones(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
        cy, cx = IMAGE_SIZE // 2, IMAGE_SIZE // 2
        half = mask_size // 2
        mask[:, :, cy - half:cy + half, cx - half:cx + half] = 0

        masked_display = (img * mask).clamp(-1, 1) * 0.5 + 0.5
        result = repaint_inpainting(model, schedule, img, mask, device)

        fig, axes = plt.subplots(1, 3, figsize=(9, 3))
        axes[0].imshow((img[0].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1))
        axes[0].set_title("Original")
        axes[0].axis("off")
        axes[1].imshow(masked_display[0].cpu().permute(1, 2, 0).numpy().clip(0, 1))
        axes[1].set_title("Masked (24×24)")
        axes[1].axis("off")
        axes[2].imshow(result[0].cpu().permute(1, 2, 0).numpy().clip(0, 1))
        axes[2].set_title("Inpainted")
        axes[2].axis("off")
        plt.tight_layout()
        plt.savefig(f"{save_dir}/example_{i + 1:03d}.png", dpi=150)
        plt.close()

    print(f"Saved {n_examples} inpainting results to {save_dir}")

run_inpainting_demo(model, sched, dataset, device)

Inpainting:   0%|          | 0/50 [00:00<?, ?it/s]

Saved 50 inpainting results to /kaggle/working/ddpm_baseline/inpainting


In [ ]:
import subprocess
try:
    import lpips
except ImportError:
    subprocess.check_call(["pip", "install", "lpips", "-q"])
    import lpips
 
@torch.no_grad()
def compute_inpainting_metrics(model, schedule, dataset, device,
                                n_examples=50, mask_size=24):
    model.eval()
 
    loss_fn = lpips.LPIPS(net='alex').to(device)
 
    mse_scores = []
    lpips_scores = []
 
    cy, cx = IMAGE_SIZE // 2, IMAGE_SIZE // 2
    half = mask_size // 2
    y0, y1 = cy - half, cy + half
    x0, x1 = cx - half, cx + half
 
    for i in tqdm(range(n_examples), desc="Inpainting metrics"):
        img = dataset[i * (len(dataset) // n_examples)].unsqueeze(0).to(device)
 
        mask = torch.ones(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
        mask[:, :, y0:y1, x0:x1] = 0
 
        result = repaint_inpainting(model, schedule, img, mask, device)
 
        # Original in [0, 1]
        original_01 = img * 0.5 + 0.5
        # result is already [0, 1]
 
        # Crop the masked region
        orig_crop = original_01[:, :, y0:y1, x0:x1]
        recon_crop = result[:, :, y0:y1, x0:x1]
 
        # MSE on masked region
        mse = F.mse_loss(recon_crop, orig_crop).item()
        mse_scores.append(mse)
 
        # LPIPS expects [-1, 1] range and minimum ~64x64
        orig_lpips = F.interpolate(orig_crop * 2 - 1, size=(64, 64), mode='bilinear', align_corners=False)
        recon_lpips = F.interpolate(recon_crop * 2 - 1, size=(64, 64), mode='bilinear', align_corners=False)
        lp = loss_fn(orig_lpips, recon_lpips).item()
        lpips_scores.append(lp)
 
    mse_arr = np.array(mse_scores)
    lpips_arr = np.array(lpips_scores)
 
    print(f"\n{'='*40}")
    print(f"Inpainting metrics (N={n_examples}, mask={mask_size}x{mask_size})")
    print(f"{'='*40}")
    print(f"MSE:   {mse_arr.mean():.6f} ± {mse_arr.std():.6f}")
    print(f"LPIPS: {lpips_arr.mean():.4f} ± {lpips_arr.std():.4f}")
 
    return mse_arr, lpips_arr
 
 
mse_scores, lpips_scores = compute_inpainting_metrics(model, sched, dataset, device)



## Style transfer

In [ ]:
!wget -q -O /kaggle/working/starry_night.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/1280px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"
!wget -q -O /kaggle/working/great_wave.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a5/Tsunami_by_hokusai_19th_century.jpg/1280px-Tsunami_by_hokusai_19th_century.jpg"

import torchvision.models as models

def gram_matrix(x):
    B, C, H, W = x.shape
    f = x.view(B, C, -1)
    return torch.bmm(f, f.transpose(1, 2)) / (C * H * W)

class StyleTransfer:
    def __init__(self, device):
        vgg = models.vgg19(pretrained=True).features.to(device).eval()
        for p in vgg.parameters():
            p.requires_grad_(False)
        self.vgg = vgg
        self.device = device
        self.content_layers = [21]
        self.style_layers = [0, 5, 10, 19, 28]

    def extract_features(self, x):
        features = {}
        for i, layer in enumerate(self.vgg):
            x = layer(x)
            if i in self.content_layers or i in self.style_layers:
                features[i] = x
        return features

    def run(self, content_img, style_img, n_steps=500, alpha=1, gamma=1e7):
        cf = self.extract_features(content_img)
        sf = self.extract_features(style_img)
        style_grams = {k: gram_matrix(v) for k, v in sf.items() if k in self.style_layers}
        target = content_img.clone().requires_grad_(True)
        optimizer = torch.optim.Adam([target], lr=0.01)
        for _ in range(n_steps):
            tf = self.extract_features(target)
            content_loss = sum(F.mse_loss(tf[l], cf[l]) for l in self.content_layers)
            style_loss = sum(F.mse_loss(gram_matrix(tf[l]), style_grams[l]) for l in self.style_layers)
            loss = alpha * content_loss + gamma * style_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        return target.detach()

vgg_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
vgg_std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
def vgg_normalize(img): return (img - vgg_mean) / vgg_std
def vgg_denormalize(img): return (img * vgg_std + vgg_mean).clamp(0, 1)

style_transfer = StyleTransfer(device)
style_transform = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])

N_CONTENT = 50
content_samples = sample_ddpm(model, sched, N_CONTENT, device)
content_large = F.interpolate(content_samples, size=(256, 256), mode='bilinear', align_corners=False)

style_paths = [
    ("/kaggle/working/starry_night.jpg", "starry_night"),
    ("/kaggle/working/great_wave.jpg",   "great_wave"),
]

save_dir = f"{OUTPUT_DIR}/style_transfer"
os.makedirs(save_dir, exist_ok=True)

for style_path, style_name in style_paths:
    style_img = style_transform(Image.open(style_path).convert("RGB")).unsqueeze(0).to(device)
    style_large = F.interpolate(style_img, size=(256, 256), mode='bilinear', align_corners=False)

    for j in range(N_CONTENT):
        print(f"{style_name} | cat {j + 1}/{N_CONTENT}")
        result = style_transfer.run(
            vgg_normalize(content_large[j:j+1]),
            vgg_normalize(style_large)
        )
        result_display = vgg_denormalize(result)
        result_small = F.interpolate(result_display, size=(64, 64), mode='bilinear', align_corners=False)

        fig, axes = plt.subplots(1, 3, figsize=(9, 3))
        axes[0].imshow(style_img[0].cpu().permute(1, 2, 0).numpy().clip(0, 1))
        axes[0].set_title("Style")
        axes[0].axis("off")
        axes[1].imshow(content_samples[j].cpu().permute(1, 2, 0).numpy().clip(0, 1))
        axes[1].set_title(f"Cat {j + 1}")
        axes[1].axis("off")
        axes[2].imshow(result_small[0].cpu().permute(1, 2, 0).numpy().clip(0, 1))
        axes[2].set_title("Result")
        axes[2].axis("off")
        plt.suptitle(f"Style Transfer - {style_name} - cat {j + 1}", fontsize=10)
        plt.tight_layout()
        plt.savefig(f"{save_dir}/{style_name}_cat_{j + 1:02d}.png", dpi=150)
        plt.close()
        print(f"  Saved {style_name}_cat_{j + 1:02d}.png")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 227MB/s]


Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

starry_night | cat 1/50
  Saved starry_night_cat_01.png
starry_night | cat 2/50
  Saved starry_night_cat_02.png
starry_night | cat 3/50
  Saved starry_night_cat_03.png
starry_night | cat 4/50
  Saved starry_night_cat_04.png
starry_night | cat 5/50
  Saved starry_night_cat_05.png
starry_night | cat 6/50
  Saved starry_night_cat_06.png
starry_night | cat 7/50
  Saved starry_night_cat_07.png
starry_night | cat 8/50
  Saved starry_night_cat_08.png
starry_night | cat 9/50
  Saved starry_night_cat_09.png
starry_night | cat 10/50
  Saved starry_night_cat_10.png
starry_night | cat 11/50
  Saved starry_night_cat_11.png
starry_night | cat 12/50
  Saved starry_night_cat_12.png
starry_night | cat 13/50
  Saved starry_night_cat_13.png
starry_night | cat 14/50
  Saved starry_night_cat_14.png
starry_night | cat 15/50
  Saved starry_night_cat_15.png
starry_night | cat 16/50
  Saved starry_night_cat_16.png
starry_night | cat 17/50
  Saved starry_night_cat_17.png
starry_night | cat 18/50
  Saved starry_